# 🧪 Creating Custom Gymnasium Environments
This tutorial covers how to build and test custom Gymnasium environments for Reinforcement Learning (RL). We'll begin with a simple 1D movement environment, then move to a more complex example simulating an Emergency Room in a hospital.

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This section performs environment registration or additional logic.


In [18]:

# Activate the python environment   as we did install it in the previous lab 
source rl-env/bin/activate  # On Windows: rl-env\Scripts\activate

SyntaxError: invalid syntax (1366157858.py, line 2)

In [ ]:
# ✅ Install gymnasium if not already installed
#!pip install gymnasium numpy
# if it is already installed, it will skip the installation step
import gymnasium as gym
import numpy as np

## 📦 SimpleEnv: 1D Navigation to a Goal
- Action space: 0 = left, 1 = right
- State space: 1D position from 0 to 10
- Goal: Reach position >= 10

![Simple ENV](./images/simple.png)

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This defines a new custom environment class inheriting from `gym.Env`.
- It sets up the action space, observation space, and environment metadata.


In [ ]:
# Import the Gymnasium library for building custom environments
import gymnasium as gym

# Import the 'spaces' module to define action and observation spaces
from gymnasium import spaces

# Import NumPy for numerical operations
import numpy as np

# Define a custom environment by inheriting from gym.Env
class SimpleEnv(gym.Env):
    # Optional metadata specifying available render modes
    metadata = {"render_modes": ["human"]}

    def __init__(self):
        # Call the constructor of the base class
        super(SimpleEnv, self).__init__()

        # Define the action space:
        # Discrete(2) means two actions: 0 (left), 1 (right)
        self.action_space = spaces.Discrete(2)

        # Define the observation (state) space:
        # A 1D position in the range [0, 10]
        self.observation_space = spaces.Box(low=0, high=10, shape=(1,), dtype=np.float32)

        # Set the goal position the agent should reach
        self.goal_position = 10.0

    def reset(self, seed=None, options=None):
        # Reset the environment state at the beginning of an episode
        # Optionally use a seed for reproducibility
        super().reset(seed=seed)

        # Initialize the agent's position randomly between 0 and 5
        self.state = np.array([np.random.uniform(0, 5)], dtype=np.float32)

        # Return the initial state and empty info dictionary
        return self.state, {}

    def step(self, action):
        # Apply the action:
        # If 0, move left (decrease position by 1)
        # If 1, move right (increase position by 1)
        if action == 0:
            self.state[0] -= 1.0
        elif action == 1:
            self.state[0] += 1.0

        # Ensure the state remains within the defined boundaries [0, 10]
        self.state = np.clip(self.state, 0.0, 10.0)

        # Check if the goal has been reached
        done = self.state[0] >= self.goal_position

        # Provide reward:
        # +10 for reaching the goal, -1 for each step otherwise
        reward = 10.0 if done else -1.0

        # Return new state, reward, done flag, truncated flag, and info
        return self.state, reward, done, False, {}

    def render(self):
        # Display the current position of the agent
        print(f"Agent Position: {self.state[0]}")

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This line creates an instance of the environment and simulates it using random actions.


In [ ]:
# Run SimpleEnv simulation
env = SimpleEnv()
obs, _ = env.reset()
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, done, _, _ = env.step(action)
    env.render()

Agent Position: 0.7311826944351196
Agent Position: 0.0
Agent Position: 1.0
Agent Position: 0.0
Agent Position: 0.0
Agent Position: 0.0
Agent Position: 1.0
Agent Position: 2.0
Agent Position: 1.0
Agent Position: 2.0
Agent Position: 3.0
Agent Position: 4.0
Agent Position: 5.0
Agent Position: 6.0
Agent Position: 7.0
Agent Position: 8.0
Agent Position: 7.0
Agent Position: 8.0
Agent Position: 7.0
Agent Position: 6.0
Agent Position: 7.0
Agent Position: 8.0
Agent Position: 7.0
Agent Position: 6.0
Agent Position: 5.0
Agent Position: 6.0
Agent Position: 7.0
Agent Position: 6.0
Agent Position: 5.0
Agent Position: 4.0
Agent Position: 5.0
Agent Position: 6.0
Agent Position: 7.0
Agent Position: 8.0
Agent Position: 9.0
Agent Position: 10.0


## 🏥 EmergencyRoomEnv: Simulating Patient Treatment
**Action Space:** `0 = treat patient`, `1 = wait`

**Observation Space:** `[patients_waiting, doctor_available]`

**Goal:** Treat all patients efficiently. Penalty for waiting too long or idle time.

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This defines a new custom environment class inheriting from `gym.Env`.
- It sets up the action space, observation space, and environment metadata.


In [ ]:
class EmergencyRoomEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(2)  # 0 = treat, 1 = wait
        self.observation_space = spaces.Box(low=0, high=10, shape=(2,), dtype=np.int32)
        self.reset()

    def reset(self, seed=None, options=None):
        self.patients = 1000
        self.doctor_available = 1
        self.time = 0
        return np.array([self.patients, self.doctor_available]), {}

    def step(self, action):
        reward = 0
        if action == 0 and self.patients > 0 and self.doctor_available:
            self.patients -= 1
            reward = 10
        elif action == 1:
            reward = -2
        else:
            reward = -5  # trying to treat with no patients or doctor

        self.time += 1
        done = self.patients == 0 or self.time >= 20
        return np.array([self.patients, self.doctor_available]), reward, done, False, {}

    def render(self):
        print(f"Time: {self.time}, Patients: {self.patients}, Doctor: {self.doctor_available}")

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This line creates an instance of the environment and simulates it using random actions.


In [ ]:
# Run EmergencyRoomEnv simulation
env = EmergencyRoomEnv()
obs, _ = env.reset()
done = False
total_reward = 0
while not done:
    action = env.action_space.sample()
    obs, reward, done, _, _ = env.step(action)
    total_reward += reward
    env.render()
print(f"Total reward: {total_reward}")

Time: 1, Patients: 999, Doctor: 1
Time: 2, Patients: 999, Doctor: 1
Time: 3, Patients: 998, Doctor: 1
Time: 4, Patients: 997, Doctor: 1
Time: 5, Patients: 996, Doctor: 1
Time: 6, Patients: 995, Doctor: 1
Time: 7, Patients: 995, Doctor: 1
Time: 8, Patients: 994, Doctor: 1
Time: 9, Patients: 993, Doctor: 1
Time: 10, Patients: 992, Doctor: 1
Time: 11, Patients: 992, Doctor: 1
Time: 12, Patients: 991, Doctor: 1
Time: 13, Patients: 991, Doctor: 1
Time: 14, Patients: 990, Doctor: 1
Time: 15, Patients: 990, Doctor: 1
Time: 16, Patients: 989, Doctor: 1
Time: 17, Patients: 988, Doctor: 1
Time: 18, Patients: 988, Doctor: 1
Time: 19, Patients: 987, Doctor: 1
Time: 20, Patients: 987, Doctor: 1
Total reward: 116
